# Training

This notebook puts everything together: data, model, loss, optimizer, and the
training loop. We'll train a classifier end-to-end on a tiny dataset.

In PyTorch you write a manual `for epoch in range(N)` loop with `loss.backward()`
and `optimizer.step()`. In idris-ml, `runTraining` handles all of that declaratively.

## Optimizers

These are C-level native optimizers. The backward pass, gradient clipping,
and parameter update happen in one fused step.

In [1]:
:t nativeSgd

Variable.nativeSgd : Double -> NativeOptimizer


In [2]:
:t nativeAdamW

Variable.nativeAdamW : Double -> Double -> Double -> Double -> Double -> Double -> NativeOptimizer


In [3]:
:t nativeRmsprop

Variable.nativeRmsprop : Double -> Double -> Double -> Double -> Double -> NativeOptimizer


## The Training Runner

`runTraining` takes four things:
1. An **epoch function** (forward + backward + optimizer step)
2. A **data source** (`IO` action that returns training data)
3. A **config** (number of epochs, early stopping)
4. The **initial model**

It returns `(trainedModel, epochsDone, finalLoss)`.

In [4]:
:t runTraining

Train.runTraining : (model -> dp -> (model, Double)) -> IO dp -> TrainConfig model -> model -> IO (model, (Nat, Double))


In [5]:
:t simpleConfig

Train.simpleConfig : Nat -> TrainConfig model


In [6]:
:t patienceConfig

Train.patienceConfig : Nat -> Nat -> TrainConfig model


## End-to-End: Train and Evaluate

Let's classify 3 points into 3 classes using a single linear layer.
This is the simplest possible training run — the idris-ml equivalent of a
PyTorch hello-world classifier.

`toTDP` converts high-level `DataPoint` values to C-level tensor data
points that survive across training epochs. After training, `toDoubleNetwork`
converts to a pure model for evaluation.

In [7]:
:exec do { srand 42;
  ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (OutputLayer ll));
  opt <- pure (nativeSgd 0.1);
  d1 <- pure (toTDP (the (DataPoint 2 3 Double) (MkDataPoint (VTensor [1.5, -2.7]) (VTensor [0, 1, 0]))));
  d2 <- pure (toTDP (the (DataPoint 2 3 Double) (MkDataPoint (VTensor [5.7, 0.0]) (VTensor [0, 0, 1]))));
  d3 <- pure (toTDP (the (DataPoint 2 3 Double) (MkDataPoint (VTensor [2.9, -1.4]) (VTensor [1, 0, 0]))));
  putStrLn ("Model: " ++ show model);
  (trained, epochs, loss) <- runTraining
    (\m, d => epochNativeTensorPre opt d crossEntropyTensor m)
    (pure [d1, d2, d3]) (simpleConfig 300) model;
  dblModel <- pure (toDoubleNetwork (emap refreshValue trained));
  putStrLn "";
  putStrLn "Predictions:";
  r1 <- pure (snd (forward dblModel (the (Vector 2 Double) (VTensor [1.5, -2.7]))));
  putStrLn ("  [1.5, -2.7] -> " ++ show r1);
  r2 <- pure (snd (forward dblModel (the (Vector 2 Double) (VTensor [5.7, 0.0]))));
  putStrLn ("  [5.7,  0.0] -> " ++ show r2);
  r3 <- pure (snd (forward dblModel (the (Vector 2 Double) (VTensor [2.9, -1.4]))));
  putStrLn ("  [2.9, -1.4] -> " ++ show r3) }

Model: Linear<2:3>
Training... [backend=tape]
  [00:00:00] 0	loss=0.2787481412352019
  [00:00:00] 100	loss=0.11785229759020249
  [00:00:00] 200	loss=0.08211732568236524
Completed in 0s (300 epochs, 0ms/epoch)

Predictions:
  [1.5, -2.7] -> [2.2016298084526476, 4.068243934487839, -2.2540680483531736]
  [5.7,  0.0] -> [1.4120846139858896, -6.058857430622522, 3.659153471409244]
  [2.9, -1.4] -> [1.7016964230152232, -0.135257387794269, 0.14812045760905357]


The model learned to classify: the highest value aligns with the correct
class for each input point.

The PyTorch equivalent would be:
```python
model = nn.Sequential(nn.Linear(2, 3), nn.Softmax(dim=-1))
optimizer = optim.SGD(model.parameters(), lr=0.1)
for epoch in range(500):
    output = model(input)
    loss = F.cross_entropy(output, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

In idris-ml, `runTraining` replaces the manual loop, and the epoch function
fuses forward + backward + optimizer step.

## Training Modes

Different architectures need different epoch functions:

| Scenario | Epoch function | Data type |
|----------|---------------|-----------|
| Classification / regression | `epochNativeTensorPre` | `TensorDataPoint` |
| RNN / LSTM / GRU sequences | `epochRecurrentNativeTensor` | `RecurrentDataPoint` |
| NTM encode-decode | `epochTwoPhaseBceNative` | `TwoPhaseDataPoint` |

The next notebook covers the recurrent case.

## Early Stopping

`simpleConfig n` runs exactly `n` epochs. For smarter stopping:

In [8]:
:t EarlyStopConfig

Train.EarlyStopConfig : Type


- `Patience n delta` — stop if loss doesn't improve by `delta` for `n` epochs
- `WindowedAvg threshold window patience` — stop when windowed average loss < threshold

`patienceConfig totalEpochs patience` is a convenient shorthand.

## Compiled Examples

For real training (thousands of epochs, larger models), use the compiled examples.
The notebook kernel buffers all output, making long runs impractical.

```bash
make example-supervised    # This notebook's task (1000 epochs)
make example-rnn           # RNN pattern prediction
make example-lstm          # LSTM sequences
make example-mnist         # CNN digit classification
make example-transformer   # Sequence sorting
make example-gpt           # Character-level language model
make example-ntm-copy      # Neural Turing Machine
make example-reinforce     # Policy gradient on CartPole
```

All accept `--epochs`, `--lr`, and `--seed` flags.

Next: [05 Sequences](05_sequences.ipynb) — recurrent models for time-series data.